In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.integrate import quad
import multiprocessing as mp

# =============================
# CONSTANTS
# =============================
G = 6.6743e-11
Msun = 1.9884e30
au = 1.496e11
fac = 1.e-3*(1/206265)*3.086e13/3.154e7



# =============================
# LOAD DATA (ROBUST)
# =============================
df = pd.read_csv("gaia_dr3_MSMS_d200pc.csv")

print("Initial dataset size:", len(df))



Initial dataset size: 81880


In [2]:
print(df.columns.tolist())

['source_id1', 'source_id2', 'R_chance', 's[kau]', 'd1[pc]', 'd1_err[pc]', 'd2[pc]', 'd2_err[pc]', 'MagG1', 'MagG2', 'M1[Msun]', 'M2[Msun]', 'mu1ra[mas/yr]', 'mu1ra_err[mas/yr]', 'mu1dec[mas/yr]', 'mu1dec_err[mas/yr]', 'mu2ra[mas/yr]', 'mu2ra_err[mas/yr]', 'mu2dec[mas/yr]', 'mu2dec_err[mas/yr]', 'RV1[km/s]', 'RV1_err[km/s]', 'RV2[km/s]', 'RV2_err[km/s]', 'gal_b[deg]', 'ruwe1', 'ruwe2', 'bp_rp1', 'bp_rp2', 'RA1[deg]', 'DEC1[deg]', 'RA2[deg]', 'DEC2[deg]', 'e', 'e0', 'e1', 'A_G_A[mag]', 'A_G_B[mag]']


In [3]:
# =============================
# INPUT
# =============================
model_choice = input("model (c/e): ")
f_multi = float(input("fraction of multiples: "))
num = input("output number: ")

model = "eccentric" if model_choice != "c" else "circular"

# =============================
# RENAME COLUMNS (CRUCIAL FIX)
# =============================
df = df.rename(columns={
    "s[kau]": "rp",
    "d1[pc]": "d_A",
    "d1_err[pc]": "d_A_err",
    "d2[pc]": "d_B",
    "d2_err[pc]": "d_B_err",
    "MagG1": "MagG_A",
    "MagG2": "MagG_B",
    "mu1ra[mas/yr]": "mux_A",
    "mu1dec[mas/yr]": "muy_A",
    "mu2ra[mas/yr]": "mux_B",
    "mu2dec[mas/yr]": "muy_B"
})

# =============================
# FILTER
# =============================
df["d_M"] = (df["d_A"]/df["d_A_err"]**2 + df["d_B"]/df["d_B_err"]**2) / \
            (1/df["d_A_err"]**2 + 1/df["d_B_err"]**2)

df = df[df["d_M"] < 200]

print("After filter:", len(df))

# =============================
# MASS (simple relation)
# =============================
def mag_to_mass(Mag):
    L = 10**(-0.4 * Mag)
    return L**(1/3.5)

df["M_A"] = mag_to_mass(df["MagG_A"])
df["M_B"] = mag_to_mass(df["MagG_B"])

# =============================
# SIMULATION
# =============================
def simulate(row):
    rp = row["rp"]
    M_A = row["M_A"]
    M_B = row["M_B"]
    d_A = row["d_A"]
    d_B = row["d_B"]

    inc = np.arccos(np.random.rand())

    eps = 0 if model == "circular" else np.random.rand()
    phi = np.random.rand() * 2*np.pi

    r = 1e3 * rp
    M = Msun * (M_A + M_B)

    vc = 1e-3 * np.sqrt(G*M/(au*r))

    dvx = -vc*np.sin(phi)
    dvy = vc*np.cos(phi)*np.cos(inc)

    mux_A = row["mux_A"] + dvx/(fac*d_A)
    muy_A = row["muy_A"] + dvy/(fac*d_A)

    mux_B = row["mux_B"] - dvx/(fac*d_B)
    muy_B = row["muy_B"] - dvy/(fac*d_B)

    return mux_A, muy_A, mux_B, muy_B

# =============================
# RUN
# =============================
if __name__ == "__main__":

    rows = df.to_dict("records")

    print("Running simulation...")
    res = list(map(simulate, rows))
    res = np.array(res)

    df["mux_A_sim"] = res[:,0]
    df["muy_A_sim"] = res[:,1]
    df["mux_B_sim"] = res[:,2]
    df["muy_B_sim"] = res[:,3]

    outfile = f"Newton_clean_{num}.csv"
    df.to_csv(outfile, index=False)

    print("Saved:", outfile)

model (c/e):  e
fraction of multiples:  0.5
output number:  1


After filter: 81088
Running simulation...
Saved: Newton_clean_1.csv
